In [1]:
import sys
sys.path.append('..')

from utilities_amigocloud import AmigocloudFunctions
from sqlalchemy import create_engine, text
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon
from shapely import wkb

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

In [13]:
# conexion a base de datos
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# obtener datos de todo catastro
def get_catastro():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None

# eliminar todas las propiedades cuyo unidad_01 este en la lista recibida
def eliminar_props(lista_cod):
    cod_string = ", ".join(map(str, lista_cod))
    query = f"""DELETE FROM dataset_396856 WHERE unidad_01 in ({cod_string})"""
    select = amigocloud.ejecutar_query_sql(ID_PROYECTO, query, 'post')
    return select

# funcion auxiliar para convertir poligono en multipoligono
def convertir_a_multipolygon(geometry):
    if isinstance(geometry, Polygon):
        return MultiPolygon([geometry])
    return geometry

# convertir poligono en formato wkb
def convertir_a_wkb(polygon):
    wkb_data = wkb.dumps(polygon, hex=True)
    return wkb_data

# cargar lotes de geodataframe a amigocloud de forma masiva
def cargar_props_a_amigocloud_bulk(gdf):
    # 1. Preparación de datos y reproyección
    gdf_gral = gdf.to_crs(epsg=4326).copy()
    # Asegurar que usamos la columna de geometría correcta
    # Si tu columna se llama 'geom', la procesamos así:
    gdf_gral['geometry_processed'] = gdf_gral['geom'].apply(convertir_a_multipolygon)
    # --- FUNCIÓN AUXILIAR DE LIMPIEZA ---
    def sql_v(val, tipo='num'):
        """
        Convierte valores de Python/Pandas a sintaxis SQL válida.
        tipo: 'num', 'str', 'date'
        """
        if pd.isna(val) or val is None or str(val).lower() == 'nan':
            return "NULL"
        if tipo == 'str':
            # Quitar espacios extras y escapar comillas simples
            texto = str(val).strip().replace("'", "''")
            return f"'{texto}'"
        if tipo == 'date':
            return f"'{val}'"
            
        # Para números (int, float)
        return str(val)
    # 2. Construir la lista de valores para el SQL
    valores_rows = []
    for index, row in gdf_gral.iterrows():
        # Generar el WKB Hexadecimal
        wkb_hex = convertir_a_wkb(row['geometry_processed'])
        # Formateamos la fila usando la función sql_v
        # El orden debe ser idéntico al INSERT del paso 4
        fila_sql = f"""(
            ST_SetSRID(ST_GeomFromWKB('\\x{wkb_hex}'), 4326),
            {sql_v(row.get('idd'))},
            {sql_v(row.get('unidad_01'))},
            {sql_v(row.get('unidad_02'), 'str')},
            {sql_v(row.get('unidad_03'))},
            {sql_v(row.get('unidad_04'), 'str')},
            {sql_v(row.get('unidad_05'), 'str')},
            {sql_v(row.get('variedad'), 'str')},
            {sql_v(row.get('soca'))},
            {sql_v(row.get('textura'), 'str')},
            {sql_v(row.get('cultivo'), 'str')},
            {sql_v(row.get('financia'), 'str')},
            {sql_v(row.get('fs'), 'date')},
            {sql_v(row.get('area'))},
            {sql_v(row.get('fc'), 'date')},
            {sql_v(row.get('zafra'))},
            {sql_v(row.get('lote_admin'), 'str')}
        )"""
        valores_rows.append(fila_sql)

    # 3. Unir todas las filas
    todos_los_valores = ",\n".join(valores_rows)
    # 4. Crear el Query Final
    insert_sql = f"""
        INSERT INTO dataset_396856
        (wkb_geometry, idd, unidad_01, unidad_02, unidad_03, unidad_04, unidad_05, variedad, soca, textura, cultivo, financia, fs, area, fc, zafra, lote_admin)
        VALUES 
        {todos_los_valores};
    """
    # 5. Ejecutar
    try:
        print(f"Enviando {len(gdf_gral)} registros a AmigoCloud (Dataset 390174)...")
        resultado = amigocloud.ejecutar_query_sql(ID_PROYECTO, insert_sql, 'post') 
        if resultado:
            print("✅ Carga masiva exitosa.")
            # Opcional: marcar como planificado localmente
            # for l_id in gdf_gral['idd']: marcar_lote_como_planificado(l_id)
        else:
            print("❌ El servidor no confirmó la carga.")
            
    except Exception as e:
        print(f"❌ Error en el Insert Masivo: {e}")
    return None

In [14]:
ID_PROYECTO = 34054
amigocloud = AmigocloudFunctions(token=API_AMIGOCLOUD_TOKEN_ADM)
amigocloud

In [15]:
# get todos los datos de catastro
gdf_catastro = get_catastro()

In [16]:
# lista de codigos de propiedad a eliminar
lista_props = [
13,
17,
47,
55,
63,
114,
173,
185,
220,
271,
275,
279,
301,
302,
303,
304,
305,
328,
391,
392,
495,
549,
550,
590,
629,
949,
1149,
1173,
1215,
1314,
1454,
1520,
1618,
1707,
1953,
2075,
2158,
2407,
239,
2358
]

In [17]:
# eliminar propiedades
eliminar_props(lista_props)

{'query': 'DELETE FROM dataset_396856 WHERE unidad_01 in (13, 17, 47, 55, 63, 114, 173, 185, 220, 271, 275, 279, 301, 302, 303, 304, 305, 328, 391, 392, 495, 549, 550, 590, 629, 949, 1149, 1173, 1215, 1314, 1454, 1520, 1618, 1707, 1953, 2075, 2158, 2407, 239, 2358)',
 'count': 1983,
 'amigo_ids': ['746ed1eb7df243ae81cfecb9b67968a7',
  '87958094cfe44e6eb3816c4b5cc67e59',
  '5497e65054e34859916622c9d63ed59c',
  '91eba070832642bca6cb59909cf46930',
  '3b2edaf43ddc45668272383ec403e25a',
  '0a065c294e784cd0b9b79b43f6b1b81d',
  '3435bc36b746470293c1c067d22f54d4',
  '0dd986e628b643d9a6e697eff19c347f',
  '003ee25dee34492196f4aedb8b5d2d4d',
  '96d1a5b94d4b44218d1c70280ee18827',
  '74a47eb4fa924005b696277f5852733b',
  '158e91b441124daaad5ae8cf02d92c7a',
  'b6bb58959a1f4fdcb9e4f3d1ba331699',
  '97ea832d25194f76954479e355c36209',
  'cde66af943b14a24b12cfd03ca84bae2',
  '9fb93e58f0c547c28e2b9e26ef786eaf',
  '4413b2e3159b475a876d8e4c44d84566',
  '1065602151a54955a1d1f3d178abaa62',
  '620dff8ee50f4b8a

In [18]:
# leccionar propiedades segun lista de codigos
props_select_segun_lista = gdf_catastro[gdf_catastro['unidad_01'].isin(lista_props)]

In [19]:
# cargar gdf a amigocloud de forma masiva
cargar_props_a_amigocloud_bulk(props_select_segun_lista)

Enviando 1078 registros a AmigoCloud (Dataset 390174)...
✅ Carga masiva exitosa.
